In [3]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

options = Options()
# Argumentos de estabilidad para Docker
options.add_argument("--headless")
options.add_argument("--no-sandbox")
options.add_argument("--disable-dev-shm-usage")
options.add_argument("--disable-gpu")

try:
    # El Manager instala el driver compatible con la versión de Chrome instalada
    driver = webdriver.Chrome(
        service=Service(ChromeDriverManager().install()), 
        options=options
    )
    
    driver.get("https://www.google.com")
    print(f"¡CONECTADO! Título de la página: {driver.title}")
    driver.quit()
    
except Exception as e:
    print(f" Error: {e}")

¡CONECTADO! Título de la página: Google


In [5]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from webdriver_manager.chrome import ChromeDriverManager

In [6]:
def iniciar_navegador():
    """
    Configura el navegador Chrome en modo headless para el scraping.
    Esta configuración está optimizada para entornos de laboratorio.
    """
    options = Options()
    options.binary_location = "/usr/bin/google-chrome"
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    
    # CORRECCIÓN: User-agent completo (antes estaba cortado)
    options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36")
    
    return webdriver.Chrome(
        service=Service(ChromeDriverManager().install()),
        options=options
    )

print("Función iniciar_navegador() definida")

Función iniciar_navegador() definida


In [7]:
#URL del sitio asignado
URL_OBJETIVO = "https://books.toscrape.com/"

# CORRECCIÓN: Selectores CSS correctos
SELECTOR_CONTENEDOR = "article.product_pod"  # Cada libro está en <article class="product_pod">
SELECTOR_DATO_A = "h3 a"                     # Título: <h3><a title="...">
SELECTOR_DATO_B = "p.price_color"            # Precio: <p class="price_color">

print("Configuración establecida:")
print(f"   URL: {URL_OBJETIVO}")
print(f"   Contenedor: {SELECTOR_CONTENEDOR}")
print(f"   Dato A (Título): {SELECTOR_DATO_A}")
print(f"   Dato B (Precio): {SELECTOR_DATO_B}")

Configuración establecida:
   URL: https://books.toscrape.com/
   Contenedor: article.product_pod
   Dato A (Título): h3 a
   Dato B (Precio): p.price_color


In [8]:
# Iniciar el navegador
driver = iniciar_navegador()
dataset_final = []

try:
    print(f"Conectando a la fuente de datos...")
    driver.get(URL_OBJETIVO)
    time.sleep(5)  # Tiempo de espera para carga de datos dinámicos
    
    # Capturamos todos los bloques de información
    elementos = driver.find_elements(By.CSS_SELECTOR, SELECTOR_CONTENEDOR)
    print(f"Se encontraron {len(elementos)} registros potenciales.")
    
    # Procesamos cada elemento encontrado
    for idx, item in enumerate(elementos, 1):
        try:
            # CORRECCIÓN #1: Extracción del título completo usando el atributo 'title'
            # El .text está truncado, pero el atributo 'title' tiene el nombre completo
            titulo_elemento = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_A)
            columna_a = titulo_elemento.get_attribute("title")
            
            # Extracción del precio
            columna_b = item.find_element(By.CSS_SELECTOR, SELECTOR_DATO_B).text
            
            # CORRECCIÓN #2: Limpieza adecuada del precio
            # Antes: "".join(filter(str.isdigit, columna_b)) eliminaba el punto decimal
            # Ahora: solo removemos el símbolo £ y espacios
            valor_limpio = columna_b.replace("£", "").strip()
            
            # Agregamos el registro al dataset
            dataset_final.append({
                "variable_nombre": columna_a,      # Título del libro
                "variable_valor": valor_limpio,    # Precio sin símbolo
                "status": 1.0                      # Indicador de registro procesado
            })
            
            # Mostrar progreso cada 5 libros
            if idx % 5 == 0:
                print(f"  ✓ Procesados {idx}/{len(elementos)} libros...")
            
        except Exception as e:
            # CORRECCIÓN #3: Manejo de errores informativo
            print(f"⚠ Error procesando elemento {idx}: {e}")
            continue
    
    print(f"\nExtracción completada: {len(dataset_final)} registros capturados")
    
finally:
    driver.quit()
    print("Navegador cerrado.")

Conectando a la fuente de datos...
Se encontraron 20 registros potenciales.
  ✓ Procesados 5/20 libros...
  ✓ Procesados 10/20 libros...
  ✓ Procesados 15/20 libros...
  ✓ Procesados 20/20 libros...

Extracción completada: 20 registros capturados
Navegador cerrado.


In [9]:
if dataset_final:
    # Crear DataFrame con los datos extraídos
    df = pd.DataFrame(dataset_final)
    
    # Guardar en CSV
    nombre_archivo = "datos_extraidos_grupo.csv"
    df.to_csv(nombre_archivo, index=False)
    
    print(f"Proceso exitoso. Archivo generado: {nombre_archivo}")
    print(f"Total de registros capturados: {len(df)}")
    print("\nMuestra de los primeros 5 datos:")
    display(df.head())
    
    # BONUS: Estadísticas básicas de los datos
    print("\nEstadísticas de precios:")
    df['precio_numerico'] = pd.to_numeric(df['variable_valor'], errors='coerce')
    print(f"  - Precio promedio: £{df['precio_numerico'].mean():.2f}")
    print(f"  - Precio mínimo: £{df['precio_numerico'].min():.2f}")
    print(f"  - Precio máximo: £{df['precio_numerico'].max():.2f}")
    print(f"  - Desviación estándar: £{df['precio_numerico'].std():.2f}")
    
else:
    print("No se capturaron datos. Revisar los selectores CSS.")

Proceso exitoso. Archivo generado: datos_extraidos_grupo.csv
Total de registros capturados: 20

Muestra de los primeros 5 datos:


,variable_nombre,variable_valor,status
0,A Light in the Attic,51.77,1.0
1,Tipping the Velvet,53.74,1.0
2,Soumission,50.10,1.0
3,Sharp Objects,47.82,1.0
4,Sapiens: A Brief History of Humankind,54.23,1.0



Estadísticas de precios:
  - Precio promedio: £38.05
  - Precio mínimo: £13.99
  - Precio máximo: £57.25
  - Desviación estándar: £15.14
